Build dim_race table 

1.Read Silver Races Table

2.Read Silver Circuits Table

3.Join Races with Circuits

4.Select Required Columns

5.Write to Gold dim_races Table

In [0]:
%run "/Workspace/Users/ganeshgadade157@gmail.com/Projects/Formula-f1/00.common/01.envioment_config"

In [0]:
from pyspark.sql.functions import col

# Table name for gold dim_races
table_name = f"{catlog_name}.{gold_schema}.dim_races"

# Step 1 - Read silver races and circuits tables
df_races    = spark.read.table("dev.silver.races")
df_circuits = spark.read.table("dev.silver.circuits")

# Step 2 - Join races with circuits on circuit_id and select required columns
df_dim_race = df_races.alias("r").join(
    df_circuits.alias("c"),
    col("r.circuit_id") == col("c.circuit_id"),
    "inner"
).select(
    col("r.season"),       # Season year
    col("r.round"),        # Round number
    col("r.race_name"),    # Name of the race
    col("r.date"),         # Race date
    col("c.circuit_name"), # Circuit name
    col("c.locality"),     # City/locality of the circuit
    col("c.country")       # Country of the circuit
)

# Step 3 - Write to gold dim_races table in delta format
df_dim_race.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(table_name)